In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

In [2]:
from datasets import load_dataset
dataset_path = "dataset/dataset_curation_agent/valid_codes.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

dataset = dataset.shuffle(seed=42)

total_len = len(dataset)
train_end = int(0.7 * total_len)
eval_end = int(0.9 * total_len)

# Slice the dataset
train_dataset = dataset.select(range(0, train_end))
eval_dataset  = dataset.select(range(train_end, eval_end))
test_dataset  = dataset.select(range(eval_end, total_len))

# Print lengths to verify
print("Total:", total_len)
print("Train:", len(train_dataset))
print("Eval:", len(eval_dataset))
print("Test:", len(test_dataset))
print("\n")

print("-------------------------------------\nTrain:","example: ",train_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nEval:", len(eval_dataset),"\n example: ",eval_dataset[0],"\n-------------------------------------\n")
print("-------------------------------------\nTest:", len(test_dataset),"\n example: ",test_dataset[0],"\n-------------------------------------\n")

/home/diego/projects/Code-Fixer-LLM-Agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total: 657
Train: 459
Eval: 194
Test: 4


-------------------------------------
Train: example:  {'task': 'Fix the issue in the following Python code.', 'buggy_code': "def __repr__(self):\n    return '<%s.%s instance at %s: %s>' * (\n        self.__class__.__module__,\n        self.__class__.__name__,\n        hex(id(self)),\n        self.command\n        )", 'correct_code': "def __repr__(self):\n    return '<%s.%s instance at %s: %s>' % (\n        self.__class__.__module__,\n        self.__class__.__name__,\n        hex(id(self)),\n        self.command\n        )", 'unit_test': 'def check(candidate):\n    # Assuming the candidate is a class with __repr__ implemented as shown.\n    \n    # Test case 1: Check if the representation includes the correct module, class name, id, and command.\n    class CommandInstance:\n        def __init__(self, command):\n            self.command = command\n        \n        __repr__ = candidate\n    \n    instance1 = CommandInstance("test_command")\n    

In [3]:
EVAL_REFERENCES = [ex["correct_code"] for ex in eval_dataset]
TEST_REFERENCES = [ex["correct_code"] for ex in test_dataset]
print("eval_references:", EVAL_REFERENCES[0],"\n")
print("test_references:", TEST_REFERENCES[0],"\n")

eval_references: def __init__(self, output_vars, *args, **kwargs):
    output_vars = self.replicate_vars(output_vars)
    _, _, replaced_vars = self._get_bn_params(output_vars)
    super(ApproxTestMonitoring, self).__init__(replaced_vars, *args,
                                               **kwargs) 

test_references: def tearDown(self):
    if hasattr(self, 'env') and hasattr(self.env, 'f_disable_logging'):
        self.env.f_disable_logging()
    self.clear_handlers()
    remove_data() 



In [4]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [5]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length = 512,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.5.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
from tqdm import tqdm
import evaluate

def evaluate_pass_at_k(model, tokenizer, prompts, references, k_values=[1, 5, 10], num_completions=10, max_new_tokens=256):
    code_eval = evaluate.load("code_eval")

    all_predictions = []

    model.eval()
    for prompt in tqdm(prompts, desc="Generating Completions"):
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
        outputs = model.generate(
            input_ids=input_ids,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=num_completions,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
        torch.cuda.empty_cache()
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        #print(f"decoded_completions: {decoded}")
        # Extract completions
        cleaned = []
        for d in decoded:
            parts = d.split("### Fixed Code:")
            cleaned.append(parts[-1].strip() if len(parts) > 1 else d.strip())

        all_predictions.append(cleaned)

    print("\n✅ All completions generated. Computing pass@k...\n")
    result, _ = code_eval.compute(
        references=references,
        predictions=all_predictions,
        k=k_values,
    )

    print("🎯 Final pass@k scores:")
    scores = {}
    for k in k_values:
        score = result.get(f'pass@{k}', 'N/A')
        if isinstance(score, (float, int)):
            print(f"pass@{k}: {score:.4f}")
            scores[f'pass@{k}'] = score
        else:
            print(f"pass@{k}: {score}")
            scores[f'pass@{k}'] = score
    
    return scores

In [8]:
prompts = []
for ex in test_dataset:
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{ex["task"]}

### Input:
{ex["buggy_code"]}

### Response:"""
    prompts.append(prompt)

pass_at_k_scores = evaluate_pass_at_k(model, tokenizer, prompts, TEST_REFERENCES)
print(pass_at_k_scores)

Generating Completions:   0%|          | 0/4 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/home/diego/projects/Code-Fixer-LLM-Agent/venv/lib/python3.12/site-packages/unsloth/kernels/utils.py:438: UserWarning: An output with one or more elements was resized since it had shape [1, 10, 2560], which does not match the required output shape [10, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:30.)
  out = torch_matmul(X, W.t(), out = out)
/home/diego/projects/Code-Fixer-LLM-Agent/venv/lib/python3.12/site-packages/unsloth/kernels/utils.py:443: UserW


✅ All completions generated. Computing pass@k...

🎯 Final pass@k scores:
pass@1: 0.0000
pass@5: 0.0000
pass@10: 0.0000
{'pass@1': np.float64(0.0), 'pass@5': np.float64(0.0), 'pass@10': np.float64(0.0)}
